# Project 02: Fashion-MNIST with Observatory Pro Diagnostics

This project uses a PyTorch classifier but all diagnostics, adaptive alerts, summaries,
and dashboards are powered by NeuroGebra Observatory Pro components.


## Dataset Source and Download Instructions

- Dataset: Fashion-MNIST (Zalando Research)
- Official source: https://github.com/zalandoresearch/fashion-mnist
- License: MIT
- Auto-download in this notebook: `torchvision.datasets.FashionMNIST(download=True)`
- Manual fallback:
  1. Download from the official source above.
  2. Put dataset files in `./data/FashionMNIST/raw/`.
  3. Re-run the data cell.


In [ ]:
%pip -q install "neurogebra[logging]" torch torchvision matplotlib


In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms

from neurogebra.logging.adaptive import AdaptiveLogger, AnomalyConfig
from neurogebra.logging.dashboard import DashboardExporter
from neurogebra.logging.epoch_summary import EpochSummarizer
from neurogebra.logging.health_warnings import AutoHealthWarnings
from neurogebra.logging.logger import LogLevel, TrainingLogger
from neurogebra.logging.tiered_storage import TieredStorage

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
data_root = Path("./data")
train_ds = datasets.FashionMNIST(data_root, train=True, download=True, transform=transforms.ToTensor())
test_ds = datasets.FashionMNIST(data_root, train=False, download=True, transform=transforms.ToTensor())

X_train = train_ds.data[:10000].float() / 255.0
y_train = train_ds.targets[:10000].long()
X_test = test_ds.data[:2000].float() / 255.0
y_test = test_ds.targets[:2000].long()

X_train = X_train.view(-1, 784)
X_test = X_test.view(-1, 784)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256)

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)


In [ ]:
class FMNISTMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        h = torch.relu(self.fc1(x))
        logits = self.fc2(h)
        return logits, h


model = FMNISTMLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

base_logger = TrainingLogger(level=LogLevel.EXPERT)
storage = TieredStorage(base_dir="./logs_fmnist_pro", write_debug=True)
dashboard = DashboardExporter(path="./logs_fmnist_pro/dashboard.html")
base_logger.add_backend(storage)
base_logger.add_backend(dashboard)

adaptive_logger = AdaptiveLogger(
    base_logger,
    config=AnomalyConfig(
        zeros_pct_threshold=55.0,
        gradient_spike_factor=4.0,
        loss_spike_pct=40.0,
        escalation_cooldown=8,
    ),
)
warnings_engine = AutoHealthWarnings()
summarizer = EpochSummarizer()

epochs = 4
adaptive_logger.on_train_start(total_epochs=epochs, total_samples=len(train_loader.dataset), batch_size=128)

train_losses = []
val_losses = []

for epoch in range(epochs):
    model.train()
    adaptive_logger.on_epoch_start(epoch)

    epoch_loss = 0.0
    epoch_acc = 0.0
    seen = 0

    for batch_idx, (xb, yb) in enumerate(train_loader):
        xb = xb.to(device)
        yb = yb.to(device)

        adaptive_logger.on_batch_start(batch_idx, epoch=epoch)

        optimizer.zero_grad()
        logits, hidden = model(xb)
        loss = criterion(logits, yb)
        loss.backward()

        grad_norms = {}
        for name, param in model.named_parameters():
            if param.grad is not None:
                gnorm = float(param.grad.data.norm().item())
                grad_norms[name] = gnorm
                adaptive_logger.on_gradient_computed(name, gnorm)

        optimizer.step()

        preds = torch.argmax(logits, dim=1)
        batch_acc = float((preds == yb).float().mean().item())
        batch_loss = float(loss.item())

        hidden_np = hidden.detach().cpu().numpy()
        zeros_pct = float((np.abs(hidden_np) < 1e-8).mean() * 100.0)

        alerts = warnings_engine.check_batch(
            epoch=epoch,
            batch=batch_idx,
            loss=batch_loss,
            gradient_norms=grad_norms,
            activation_stats={
                "hidden": {
                    "activation_type": "relu",
                    "zeros_pct": zeros_pct,
                    "saturation_pct": float((np.abs(hidden_np) > 0.99).mean() * 100.0),
                }
            },
        )
        for alert in alerts:
            adaptive_logger.on_health_check(
                check_name=alert.rule_name,
                severity=alert.severity,
                message=alert.message,
                recommendations=alert.recommendations,
            )

        summarizer.record_batch(
            epoch=epoch,
            metrics={"loss": batch_loss, "accuracy": batch_acc},
            gradient_norms=grad_norms,
            activation_stats={"hidden": {"zeros_pct": zeros_pct}},
        )

        adaptive_logger.on_batch_end(batch_idx, epoch=epoch, loss=batch_loss, metrics={"accuracy": batch_acc})

        epoch_loss += batch_loss * xb.size(0)
        epoch_acc += batch_acc * xb.size(0)
        seen += xb.size(0)

    model.eval()
    val_loss = 0.0
    val_seen = 0
    val_correct = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits, _ = model(xb)
            vloss = criterion(logits, yb)
            val_loss += float(vloss.item()) * xb.size(0)
            val_seen += xb.size(0)
            val_correct += int((torch.argmax(logits, dim=1) == yb).sum().item())

    train_loss = epoch_loss / seen
    train_acc = epoch_acc / seen
    val_loss_epoch = val_loss / val_seen
    val_acc_epoch = val_correct / val_seen

    train_losses.append(train_loss)
    val_losses.append(val_loss_epoch)

    epoch_summary = summarizer.finalize_epoch(epoch)
    print(epoch_summary.format_text())

    adaptive_logger.on_epoch_end(
        epoch,
        metrics={
            "loss": train_loss,
            "accuracy": train_acc,
            "val_loss": val_loss_epoch,
            "val_accuracy": val_acc_epoch,
        },
    )

adaptive_logger.on_train_end(final_metrics={"loss": train_losses[-1], "val_loss": val_losses[-1]})
dashboard_path = dashboard.save()
storage.flush()
storage.close()

print("Anomaly summary:", adaptive_logger.get_anomaly_summary())
print("Tiered storage summary:", storage.summary())
print("Dashboard:", dashboard_path)


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label="train_loss")
plt.plot(val_losses, label="val_loss")
plt.title("Fashion-MNIST with Observatory Pro")
plt.legend()
plt.show()
